#36615 - Final Project Part 3 - Delay Prediction Spark ML
## Team Polyhymnia
### Vraj Patel, Jay Louissant, Elaine Yin

Goal: Use Spark ML with engineered features to predict
whether a flight will be delayed (including cancellations).

We implement several optimizations:
1. **Date-based split** (avoids global sort and window functions).
2. **Cache + materialize once** after expensive steps.
3. **Fit preprocessing/scaling on train only** (prevents leakage).
4. **Tune on a sample first** (faster hyperparameter search), then refit on full data.

In [0]:
# Imports

import pyspark.sql.functions as F

from pyspark.ml.feature import RFormula
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

import pandas as pd


#### 1. Load feature-engineered data

In [0]:
df = spark.table("lsd_2026.default.polyhymnia_feature_engineered")

print(f"Rows: {df.count():,}")
df.printSchema()

#### 2. Define target label: departure delayed or not

In [0]:
df = df.withColumn(
    "dep_delayed",
    F.when((F.col("DEP_DELAY") > 0) | (F.col("CANCELLED") == 1), F.lit(1)).otherwise(F.lit(0))
)

df.groupBy("dep_delayed").count().show()

#### 3. Train / Test / Validation split strategy

Because we used rolling/time-based features (previous hour, past 7 days), we will not perform a fully random split. Instead, we will sort by FL_DATE (and CRS_DEP_TIME) and use the earliest 60% as the training set and next 20% as test set (for model selection / tuning). Final 20% will be used as final validation (held out).

### Why date-based?
- It matches the real use case: **train on the past, predict the future**
- It avoids the very expensive `row_number()` over a global sort

We pick cutoffs that roughly mimic a 60/20/20 split.


In [0]:
# Inspect date range
date_bounds = df.select(F.min("FL_DATE").alias("min_date"), F.max("FL_DATE").alias("max_date")).collect()[0]
print(f"Date range: {date_bounds['min_date']} to {date_bounds['max_date']}")

In [0]:
train = df.filter(F.col("FL_DATE") < "2015-01-01")

test = df.filter(
    (F.col("FL_DATE") >= "2015-01-01") &
    (F.col("FL_DATE") <  "2017-01-01")
)

valid = df.filter(F.col("FL_DATE") >= "2017-01-01")

# Cache splits because we will reuse them many times
train = train.cache()
test  = test.cache()
valid = valid.cache()

# Materialize caches once
train_n = train.count()
test_n  = test.count()
valid_n = valid.count()

print(f"Train rows: {train_n:,}")
print(f"Test rows:  {test_n:,}")
print(f"Valid rows: {valid_n:,}")


#### 4. Create an Rformula to generate model matrix

RFormula:
- Converts categorical columns into encoded features
- Produces a single `features` vector column + a numeric `label`

**Important:** We fit this on **train only** to avoid leakage.

In [0]:
formula_str = """
dep_delayed ~ 
    OP_CARRIER + ORIGIN + DEST +
    CRS_DEP_TIME + CRS_ARR_TIME + CRS_ELAPSED_TIME + DISTANCE +
    day_of_week + hour_of_day +
    crs_dep_time_bucket + arr_time_bucket +
    rate_weather_delay_dep + rate_weather_delay_arr +
    dep_traffic_z_score + carrier_delay_rate_7d + route_delay_rate_7d
"""

rf = RFormula(
    formula=formula_str,
    featuresCol="features",  # final feature vector column
    labelCol="label",
    handleInvalid="skip"     
)

## 5. Fit preprocessing on TRAIN only and transform splits
This prevents train/test leakage and is usually faster than fitting on full data.
We also cache the transformed matrices because model tuning reuses them.

In [0]:
# Fit only on train (prevents leakage)
rf_model = rf.fit(train)

# Transform each split
train_fe = rf_model.transform(train).select("label", "features")
test_fe  = rf_model.transform(test).select("label", "features")
valid_fe = rf_model.transform(valid).select("label", "features")

# Cache because we will reuse them many times in tuning + evaluation
train_fe = train_fe.cache()
test_fe  = test_fe.cache()
valid_fe = valid_fe.cache()

# Materialize caches once
_ = train_fe.count()
_ = test_fe.count()
_ = valid_fe.count()

print("Feature matrices ready.")


## 6. Random Forest model + tuning strategy

**No StandardScaler:** tree-based models do not require feature scaling.

### Optimization: Tune on a small sample first
We tune hyperparameters on a small subset of training data to reduce runtime,
then refit the best configuration on full train+test.

In [0]:
# Take a small sample for faster tuning

# Adjust fraction depending on speed needs
train_sample = train_fe.sample(withReplacement=False, fraction=0.02, seed=42).cache()
_ = train_sample.count()


In [0]:
# Define RandomForestClassifier
rf_clf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=42,
)

# Hyperparameter grid
paramGrid = (ParamGridBuilder()
    .addGrid(rf_clf.numTrees, [20, 50])      
    .addGrid(rf_clf.maxDepth, [5, 10])        
    .addGrid(rf_clf.maxBins, [16, 32])        # bins for handling categorical splits
    .build()
)

In [0]:
# Evaluator

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

In [0]:
tvs = (TrainValidationSplit()
    .setEstimator(rf_clf)
    .setEvaluator(accuracy_eval)   # tune based on accuracy
    .setEstimatorParamMaps(paramGrid)
    .setTrainRatio(0.75)
)

tvs_model = tvs.fit(train_sample)

best_rf_model = tvs_model.bestModel

print("Best model parameters:")
print(best_rf_model.explainParams())

## Consider changing this to tune using F1 instead of accuracy

## 7. Evaluate tuned model on test set

In [0]:
test_preds = best_rf_model.transform(test_fe)

test_accuracy = accuracy_eval.evaluate(test_preds)
test_f1 = f1_eval.evaluate(test_preds)

print(f"TEST Accuracy: {test_accuracy}")
print(f"TEST F1:       {test_f1}")

#### 8. Refit on train + test and evaluate on validation

In [0]:
# Combine train + test
train_test = train_fe.union(test_fe).cache()
_ = train_test.count()

# Extract best parameters
best_numTrees = best_rf_model.getNumTrees
best_maxDepth = best_rf_model.getOrDefault("maxDepth")
best_maxBins  = best_rf_model.getOrDefault("maxBins")

# Define final Random Forest model using best parameters
rf_best = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=123,
    numTrees=best_numTrees,
    maxDepth=best_maxDepth,
    maxBins=best_maxBins
)

# Fit final model on combined train + test data
final_model = rf_best.fit(train_test)

# Evaluate final model on held-out validation set
valid_preds = final_model.transform(valid_fe)


# Compute evaluation metrics
valid_accuracy = accuracy_eval.evaluate(valid_preds)
valid_f1 = f1_eval.evaluate(valid_preds)

print(f"VALIDATION Accuracy: {valid_accuracy}")
print(f"VALIDATION F1:       {valid_f1}")

#### 9. Confusion matrix derived metrics on held out validation set

In [0]:
# Confusion Matrix
cm = (valid_preds
      .groupBy("label", "prediction")
      .count()
      .toPandas())

print("Confusion Matrix (Validation):")
print(cm)

# True Positive
TP = cm[(cm["label"] == 1.0) & (cm["prediction"] == 1.0)]["count"].sum()

# True Negative
TN = cm[(cm["label"] == 0.0) & (cm["prediction"] == 0.0)]["count"].sum()

# False Positive
FP = cm[(cm["label"] == 0.0) & (cm["prediction"] == 1.0)]["count"].sum()

# False Negative
FN = cm[(cm["label"] == 1.0) & (cm["prediction"] == 0.0)]["count"].sum()

TP, TN, FP, FN = map(float, [TP, TN, FP, FN])

accuracy = (TP + TN) / (TP + TN + FP + FN)
tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

print("\nValidation Metrics:")
print(f"Accuracy:     {accuracy:.4f}")
print(f"F1 Score:     {valid_f1:.4f}")
print(f"TPR/Recall:   {tpr:.4f}")
print(f"False Positive Rate:          {fpr:.4f}")
print(f"Specificity:  {specificity:.4f}")

#### 10. Metrics for baseline model - always predict "no delay"

In [0]:
baseline = valid_fe.withColumn("prediction", F.lit(0.0))

baseline_cm = (baseline
               .groupBy("label", "prediction")
               .count()
               .toPandas())

print("Baseline Confusion Matrix:")
print(baseline_cm)

B_TP = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_TN = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()
B_FP = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_FN = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()

B_acc = (B_TP + B_TN) / (B_TP + B_TN + B_FP + B_FN)
B_tpr = B_TP / (B_TP + B_FN) if (B_TP + B_FN) > 0 else 0
B_fpr = B_FP / (B_FP + B_TN) if (B_FP + B_TN) > 0 else 0
B_spec = B_TN / (B_TN + B_FP) if (B_TN + B_FP) > 0 else 0

print("\nBaseline Metrics:")
print(f"Accuracy:     {B_acc:.4f}")
print(f"TPR/Recall:   {B_tpr:.4f}")
print(f"True Negative Rate:  {B_spec:.4f}")

#### 11. Comparison Summary

In [0]:
print("\n=== Validation Performance Comparison (Best RF vs Baseline) ===")
print(f"RF Accuracy vs Baseline: {accuracy:.4f} vs {B_acc:.4f}")
print(f"RF TPR vs Baseline:      {tpr:.4f} vs {B_tpr:.4f}")
print(f"RF FPR vs Baseline:      {fpr:.4f} vs {B_fpr:.4f}")
print(f"RF Specificity vs Base:  {specificity:.4f} vs {B_spec:.4f}")
print(f"RF F1 vs (baseline N/A): {valid_f1:.4f}")


The model appears to be very cautious when predicting delays. It flags relatively few flights as delayed, which means it rarely raises false alarms. However, this caution comes at a cost: it misses a large share of the flights that are actually delayed. In other words, when the model does predict a delay it is often correct, but it fails to identify many real delays.

#### 12. Further Investigation

In [0]:
# Manually compute positive-class (delayed = 1) metrics


# Precision for delayed flights:
# Of all flights predicted as delayed, how many were actually delayed?
precision_pos = TP / (TP + FP) if (TP + FP) > 0 else 0

# Recall (True Positive Rate):
# Of all actual delayed flights, how many did we correctly catch?
recall_pos = TP / (TP + FN) if (TP + FN) > 0 else 0

# Positive-class F1:
# Harmonic mean of precision and recall for delayed flights only
f1_pos = (
    2 * (precision_pos * recall_pos) / (precision_pos + recall_pos)
    if (precision_pos + recall_pos) > 0
    else 0
)

print("Manual Positive-Class Metrics (Delayed Flights = 1)")
print(f"Precision (Delay): {precision_pos:.4f}")
print(f"Recall (Delay):    {recall_pos:.4f}")
print(f"F1 (Delay only):   {f1_pos:.4f}")
print(f"Spark Weighted F1: {valid_f1:.4f}")

Spark’s MulticlassClassificationEvaluator computes a weighted F1 score across both classes, not the F1 score specifically for delayed flights. Because on-time flights make up the majority of the dataset, strong performance on that majority class can inflate the overall weighted F1 score.

To better understand what was happening, we manually computed precision, recall, and F1 for the positive class (delayed flights) using the confusion matrix. This isolates performance on the class that matters most operationally.

The model has a precision of 0.7311, meaning that when it predicts a flight will be delayed, it is correct about 73% of the time. In other words, it does not raise many false alarms.

However, the recall is only 0.1850, meaning the model catches just 18.5% of all actual delayed flights. It is missing more than 80% of delays. This confirms that the model is very conservative in predicting delays.

Because F1 balances precision and recall, the manually computed F1 for delayed flights is 0.2953, which is relatively low. This contrasts sharply with Spark’s reported weighted F1 of 0.6152. The higher weighted F1 occurs because Spark averages performance across both classes, and the model performs much better on the majority class (on-time flights).

The model is good at avoiding false delay predictions, but it is not very effective at identifying delayed flights.